# Modeling

Packages and setup

In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
import itertools
import math
import os
import re
from pathlib import Path
import tabulate
from IPython.display import display, Markdown
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tsa.arima.model import ARIMA
from sklearn.preprocessing import StandardScaler

from tools.coverage_functions import plot_time_series, plot_time_series_subset


# Set directory to project root
def find_project_root(start: Path = Path().absolute()) -> Path:
    for parent in start.parents:
        if (parent / "requirements.txt").exists(): return parent
    return start 
os.chdir(find_project_root())

# Preemptively set new Pandas option, also set matplotlib to close
pd.options.mode.copy_on_write = True
%matplotlib inline
%config InlineBackend.close_figures=True

# Allow reloading of custom Python classes without resetting kernel
pd.set_option('display.max_rows', 100)
%load_ext autoreload
%autoreload 2

# Load formatted data
%store -r static_data_merged
%store -r sales_data_merged

# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"2_palate_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()
    %store static_data_merged


# Data already exists
else:
    static_data = static_data_merged.copy()

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"2_palate_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()
    %store sales_data_merged

# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()

# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

# Timezones
timezones = pd.read_csv('data/timezones.csv', index_col='location_id')['timezone'].to_dict()
for loc_id, df in sales_and_menu_data.items():
    df.index = df.index.tz_convert(timezones[loc_id])
    sales_and_menu_data[loc_id] = df

# True promos
before_after_details_true = pd.read_csv('data/4_palate_data_parquet_relabeled/before_after_details_true.csv', index_col='location_id')

# Restaurants by coverage
restaurants_by_4m_coverage = pd.read_csv('data/4_palate_data_parquet_relabeled/restaurants_by_4m_coverage.csv')['location_id'].tolist()

# New locations
locations = pd.read_csv('data/4_palate_data_parquet_relabeled/locations.csv', index_col='location_id')

# Remove bad data
del sales_and_menu_data['AQD04SM0J92WA']

del sales_and_menu_data['LBMCPAYT7W36V']
del sales_and_menu_data['L3XS7WSJ4AJA3']
del sales_and_menu_data['1G5AJ17XCH2A8']
del sales_and_menu_data['3AXDVZJYN9DRS']
del sales_and_menu_data['MS8R16DY0JQAM']

del sales_and_menu_data['N0PC58FB2XAZ3']
del sales_and_menu_data['ADPFRN3QZRCXK']
del sales_and_menu_data['WJA3YCD4QBWRX']
del sales_and_menu_data['0RJH3FFPYBPEY']
del sales_and_menu_data['LZ5MR1TS37E7W']

sales_and_menu_data['VLZX7K2M9QD4T'] = pd.read_parquet('data/4_palate_data_parquet_relabeled/consolidated/VLZX7K2M9QD4T.parquet')

location_ids = list(sales_and_menu_data.keys())
before_after_details_true = before_after_details_true.loc[restaurants_by_4m_coverage]
location_ids_by_coverage = restaurants_by_4m_coverage

sales_and_menu_data['SAFK7ND1HR6XS'] = sales_and_menu_data['SAFK7ND1HR6XS'].loc['2019-04-18':'2020-03-25']
sales_and_menu_data['2HRX9P6HKXA8V'] = sales_and_menu_data['2HRX9P6HKXA8V'].loc['2019-01-01':]

Add promo exposure indicator

In [ ]:
# for loc_id, df in sales_menu_customers_data.items():
#     df = df.tz_localize(None)
#     df.loc[:,'promo'] = 0
#     df.loc[:before_after_details.loc[loc_id,'cross_over_date'], 'promo'] = 1

Putting in customer data

In [ ]:
# # Initialize dict all data
# sales_menu_customers_data = {}
# for loc_id, df in sales_and_menu_data.items():

#     # Prevent overwriting
#     df = df.copy()

#     # Keep the index, since merges don't keep it
#     df.reset_index(inplace=True)

#     # Double check customers are unique
#     customers.dropna(subset=['customer_id'], inplace=True)
#     customers.drop_duplicates(subset=['location_id', 'customer_id'], inplace=True)

#     # Combine
#     merged = pd.merge(df, customers, on=['location_id', 'customer_id'], how='left')

#     # Reset the index back to datetimes
#     merged.set_index('created_at', inplace=True, drop=False)

#     # Save
#     sales_menu_customers_data[loc_id] = merged

Roll data

In [ ]:
def season_from_month(month):
    return 'winter' if month in [12, 1, 2] else \
           'spring' if month in [3, 4, 5] else \
           'summer' if month in [6, 7, 8] else 'fall'

def weighted_avg(window):
    return (window['item_quantity'] * window['unit_price']).sum() / window['item_quantity'].sum()

def rolling_window_avg(df_, filter_col, column, name, lookback_period, lookback_unit):
    df = (df_
          #.query(filter)
          .assign(weighted_val = lambda df: df[filter_col] * df[column])
          ['weighted_val']
          .rolling(f'{lookback_period}{lookback_unit}')
          .sum()
          .rename(name)
          .to_frame()
          .reset_index()
          .drop_duplicates('created_at')
          .set_index('created_at')
          .shift(1)
          .ffill()
          .bfill()
          [name]
          )
    return df

In [ ]:
lookback_period = 1
lookback_unit = 'D'

hour_mapping = {
    22: -1, 23: -1, 
    1: -1, 6: -1, 7: -1
}

model_df_list = []
for loc_id in location_ids_by_coverage:
    df = sales_and_menu_data[loc_id]
    
    df = df.copy()
    
    if loc_id == 'VLZX7K2M9QD4T':
        df = pd.read_parquet('data/4_palate_data_parquet_relabeled/consolidated/VLZX7K2M9QD4T.parquet')
    
    if loc_id == 'SRQS8F7JWA9MZ':
        df = pd.read_parquet('data/4_palate_data_parquet_relabeled/consolidated/SRQS8F7JWA9MZ_sales_and_menu.parquet')
    
    if loc_id == '2HRX9P6HKXA8V':
        df = pd.read_parquet('data/4_palate_data_parquet_relabeled/consolidated/2HRX9P6HKXA8V_sales_and_menu.parquet')
        
    if loc_id == 'JHDN7CF1C03X5':
        df = pd.read_parquet('data/4_palate_data_parquet_relabeled/consolidated/JHDN7CF1C03X5_sales_and_menu.parquet')
    
    if loc_id not in ['VLZX7K2M9QD4T',
                      'SRQS8F7JWA9MZ', 
                      '2HRX9P6HKXA8V',
                      'JHDN7CF1C03X5']:
        df = df.query('is_plant_based != "Unsure"')
        df['vegan'] = (df['is_plant_based'] == 'Yes')
        df['vegetarian'] = (df['is_plant_based'] == 'Yes')
        df['meat'] = (df['is_plant_based'] == 'No')
        df['item_quantity'] = df['item_quantity'].round().astype(int)
        df = df.query('item_type != "Drink" and dish_category != "Alcohol"')    

    model_df = (df
                .assign(
                    hour_of_day = lambda df: df.index.to_series().dt.hour.replace(hour_mapping).astype("category"),
                    day_of_week = lambda df: df.index.to_series().dt.dayofweek.astype("category"),
                    weekend = lambda df: pd.Series(df.index.dayofweek.isin([5, 6]).astype(int), index=df.index).astype("category"),
                    meal_period = lambda df: pd.cut(df.index.to_series().dt.hour.astype("category"), 
                                        bins=[0, 5, 11, 16, 22, 24], 
                                        labels=['Late', 'Breakfast', 'Lunch', 'Dinner', 'Late'], 
                                        right=False,
                                        ordered=False).astype(str).replace({'Late': 'Dinner'}).astype("category"),
                    day_of_month = lambda df: df.index.to_series().dt.day.astype("category"),
                    month_cat = lambda df: df.index.to_series().dt.month.astype("category"),
                    month = lambda df: df.index.to_series().dt.month.astype("category").cat.codes,
                    season = lambda df: df.index.month.map(season_from_month).astype("category"),
                    year_cat = lambda df: df.index.to_series().dt.year.astype("category"),
                    year = lambda df: df.index.to_series().dt.year.astype("category").cat.codes,
                    date = lambda df: df.index.to_series().dt.date.astype("category").cat.codes)
                .reset_index()
                .rename(columns={'created_at':'created_at_tz'})
                .assign(created_at_tz = lambda df: df['created_at_tz'].dt.tz_convert('UTC'))
                .rename(columns={'created_at_tz':'created_at'})
                .set_index('created_at')
                .join([rolling_window_avg(df, 'meat', 'item_price', 'meat_window_price', lookback_period, lookback_unit),
                       rolling_window_avg(df, 'meat', 'item_quantity', 'meat_window_quantity', lookback_period, lookback_unit),
                       rolling_window_avg(df, 'vegetarian', 'item_price', 'vegetarian_window_price', lookback_period, lookback_unit),
                       rolling_window_avg(df, 'vegetarian', 'item_quantity', 'vegetarian_window_quantity', lookback_period, lookback_unit),
                       rolling_window_avg(df, 'vegan', 'item_price', 'vegan_window_price', lookback_period, lookback_unit),
                       rolling_window_avg(df, 'vegan', 'item_quantity', 'vegan_window_quantity', lookback_period, lookback_unit)],
                      how='left')
                .assign(
                    vegan_window_avg = lambda df: (df['vegan_window_price'] / df['vegan_window_quantity']).ffill().bfill(),
                    vegetarian_window_avg = lambda df: (df['vegetarian_window_price'] / df['vegetarian_window_quantity']).ffill().bfill(),
                    meat_window_avg = lambda df: (df['meat_window_price'] / df['meat_window_quantity']).ffill().bfill(),
                    vegan_outcome = lambda df: 1*df['vegan'],
                    nonvegan_outcome = lambda df: 1 - df['vegan_outcome'],
                    vegetarian_outcome = lambda df: 1*df['vegetarian'],
                    meat_outcome = lambda df: 1 - df['vegetarian_outcome'])
                #.pipe(lambda df: print(df['item_quantity'].sum()) or df)
                #.pipe(lambda df: print(loc_id) or df)
                #.pipe(lambda df: print(df.shape) or df)
                .reset_index()
                .loc[lambda df: df.index.repeat(df['item_quantity'])]
                .set_index('created_at')
                .assign(item_quantity = 1)
                #.pipe(lambda df: print(df['item_quantity'].sum()) or df)
                #.pipe(lambda df: print(df.shape) or df)
                )
    #if loc_id in ['SRQS8F7JWA9MZ', '2HRX9P6HKXA8V']:
    model_df_list.append(model_df)
    
model_data = pd.concat(model_df_list).assign(location_id = lambda df: df['location_id'].astype('category'))
# model_data.to_parquet('data/5_palate_data_parquet_modeling/all_locations.parquet')

Customer Deviations

In [ ]:
before_after_customers = pd.read_pickle('data/before_after_customers.pkl').set_index('loc_id')['customer_ids'].sum()
repeat_customers = model_data.groupby('customer_id')['item_name'].size().to_frame(name='count').query('count > 1').index.tolist()
customer_orders_day = (model_data
                       .assign(temp_date = lambda df: df.index.date)
                       .query('customer_id.isin(@before_after_customers)')
                       .groupby(['customer_id','temp_date'])
                       ['nonvegan_outcome']
                       .sum())
customer_mean_day = customer_orders_day.groupby('customer_id').transform('mean').round().astype(int)
deviations = customer_orders_day.sub(customer_mean_day)#.value_counts().sort_index()
deviations.plot(kind='hist', bins=800, edgecolor='white')
#plt.bar(x=deviations.index, height=deviations)
plt.title('Customer Orders Day Deviation from Their Means')
plt.xlabel('Nonvegan Deviations from Their Mean Orders')
#plt.xticks(ticks=range(-30,40,10))
plt.xlim(-20,20)
plt.show()

In [ ]:
customers

In [ ]:
model_data.columns

In [ ]:
import math
restaurants_by_4m_coverage.remove('VLZX7K2M9QD4T')
before_after_customers_by_loc = (pd.read_pickle('data/before_after_customers.pkl')
                                 .set_index('loc_id')
                                 ['customer_ids']
                                 .loc[restaurants_by_4m_coverage])
restaurants_by_4m_coverage.insert(0, 'VLZX7K2M9QD4T')
ncols = 4
nrows = math.ceil(len(before_after_customers_by_loc) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(20, 5 * nrows))
axes = axes.flatten()
for i, (loc_id, customers_in_loc) in enumerate(before_after_customers_by_loc.items()):
    
    ax = axes[i]
    if not customers_in_loc:
        ax.set_title(f"Location {loc_id}\nNo customers found.")
        ax.axis('off') # Turn off the axis if no data.
        continue

    customer_orders_day = (model_data
                           .assign(temp_date=lambda df: df.index.date)
                           .query('customer_id.isin(@customers_in_loc)')
                           .groupby(['customer_id', 'temp_date'])
                           ['nonvegan_outcome']
                           .sum())

    if customer_orders_day.empty:
        ax.set_title(f"Location {loc_id}\nNo order data found.")
        ax.axis('off') # Turn off the axis if no data.
        continue

    customer_mean_day = customer_orders_day.groupby('customer_id').transform('mean').round().astype(int)
    deviations = customer_orders_day.sub(customer_mean_day)
    deviation_counts = deviations.value_counts().sort_index()

    ax.bar(deviation_counts.index, deviation_counts.values, color='skyblue', edgecolor='black')
    ax.set_title(f'Location ID: {loc_id}')
    ax.set_xlabel('Deviations from Mean')
    ax.set_ylabel('Frequency')
    ax.set_xlim(-20.5, 20.5)
    ax.set_xticks(ticks=range(-20, 21, 5)) # Adjusted ticks for smaller subplot size
    ax.grid(axis='y', linestyle='--', alpha=0.7)

for j in range(i + 1, len(axes)):
    axes[j].axis('off')
fig.suptitle('Customer Orders Day Deviation from Their Means by Location', fontsize=24, fontweight='bold')
plt.tight_layout(rect=[0, 0.03, 1, 0.96])
plt.show()

In [ ]:
restaurants_by_4m_coverage.remove('VLZX7K2M9QD4T')

before_after_customers_by_loc = (pd.read_pickle('data/before_after_customers.pkl')
                                   .set_index('loc_id')
                                   ['customer_ids']
                                   .loc[restaurants_by_4m_coverage])

restaurants_by_4m_coverage.insert(0, 'VLZX7K2M9QD4T')

ncols = 4
nrows = math.ceil(len(before_after_customers_by_loc) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(20, 6 * nrows))
axes = axes.flatten()

# --- MODIFICATION: Create a colormap for the 1-10 purchase range ---
min_purchases = 1
max_purchases = 8
num_colors = max_purchases - min_purchases + 1
colors = plt.cm.viridis(np.linspace(0, 1, num_colors))

for i, (loc_id, customers_in_loc) in enumerate(before_after_customers_by_loc.items()):
    
    ax = axes[i]
    if not customers_in_loc:
        ax.set_title(f"Location {loc_id}\nNo customers found.")
        ax.axis('off')
        continue

    customer_orders_day = (model_data
                           .assign(temp_date=lambda df: df.index.date)
                           .query('customer_id.isin(@customers_in_loc)')
                           .groupby(['customer_id', 'temp_date'])
                           ['nonvegan_outcome']
                           .sum())

    if customer_orders_day.empty:
        ax.set_title(f"Location {loc_id}\nNo order data found.")
        ax.axis('off')
        continue
    
    # Prepare data for stacked bar chart
    deviation_df = pd.DataFrame({'orders': customer_orders_day})
    deviation_df['mean'] = deviation_df.groupby('customer_id')['orders'].transform('mean').round().astype(int)
    deviation_df['deviation'] = deviation_df['orders'] - deviation_df['mean']
    
    # Group by both deviation and the original number of orders to get the counts for each segment.
    stacked_data = deviation_df.groupby(['deviation', 'orders']).size().unstack(fill_value=0)

    # Create the stacked bar plot
    bottom = np.zeros(len(stacked_data))

    # Loop through each original order count to create the stacks.
    for order_count, counts_per_deviation in stacked_data.items():
        # --- MODIFICATION: Only plot bars for purchase counts between 1 and 10 ---
        if min_purchases <= order_count <= max_purchases:
            # Map the order count (1-10) to a color index (0-9)
            color_index = order_count - min_purchases
            color = colors[color_index]
            ax.bar(stacked_data.index, counts_per_deviation, bottom=bottom, label=f'{order_count} orders', color=color, edgecolor='white', linewidth=0.7)
            bottom += counts_per_deviation.values

    ax.set_title(f'Location ID: {loc_id}')
    ax.set_xlabel('Deviations from Mean')
    ax.set_ylabel('Frequency')
    ax.set_xlim(-15.5, 15.5)
    ax.set_xticks(ticks=range(-15, 16, 5))
    ax.grid(axis='y', linestyle='--', alpha=0.3)

# Add a single, shared legend for the entire figure
handles, labels = ax.get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, title='Original Purchases', loc='center right', bbox_to_anchor=(1.05, 0.5))

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

fig.suptitle('Customer Orders Day Deviation from Their Means by Location', fontsize=24, fontweight='bold')
plt.tight_layout(rect=[0, 0.03, 0.95, 0.96])
plt.show()

In [ ]:
restaurants_by_4m_coverage.remove('VLZX7K2M9QD4T')

before_after_customers_by_loc = (pd.read_pickle('data/before_after_customers.pkl')
                                   .set_index('loc_id')
                                   ['customer_ids']
                                   .loc[restaurants_by_4m_coverage])

restaurants_by_4m_coverage.insert(0, 'VLZX7K2M9QD4T')

ncols = 4
nrows = math.ceil(len(before_after_customers_by_loc) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(20, 6 * nrows))
axes = axes.flatten()

# --- MODIFICATION: Define colors for gender ---
gender_colors = {'male': 'cornflowerblue', 'female': 'lightcoral'}

for i, (loc_id, customers_in_loc) in enumerate(before_after_customers_by_loc.items()):
    
    ax = axes[i]
    if not customers_in_loc:
        ax.set_title(f"Location {loc_id}\nNo customers found.")
        ax.axis('off')
        continue

    customer_orders_day = (model_data
                           .assign(temp_date=lambda df: df.index.date)
                           .query('customer_id.isin(@customers_in_loc)')
                           .groupby(['customer_id', 'temp_date'])
                           ['nonvegan_outcome']
                           .sum())

    if customer_orders_day.empty:
        ax.set_title(f"Location {loc_id}\nNo order data found.")
        ax.axis('off')
        continue
    
    # --- MODIFICATION: Prepare data for gender-stacked bar chart ---
    # Convert series to DataFrame and reset index to get customer_id as a column.
    deviation_df = customer_orders_day.to_frame(name='orders').reset_index()
    
    # Merge with the customers DataFrame to get gender information.
    # Assuming 'customers' DataFrame has 'customer_id' and 'gender' columns.
    deviation_df = pd.merge(deviation_df, customers, on='customer_id', how='left')

    # Calculate mean and deviation after merging.
    deviation_df['mean'] = deviation_df.groupby('customer_id')['orders'].transform('mean').round().astype(int)
    deviation_df['deviation'] = deviation_df['orders'] - deviation_df['mean']
    
    # Group by deviation and gender to get counts for stacking.
    stacked_data = deviation_df.groupby(['deviation', 'gender']).size().unstack(fill_value=0)
    
    # Ensure both Male and female columns exist to avoid errors.
    if 'male' not in stacked_data: stacked_data['male'] = 0
    if 'female' not in stacked_data: stacked_data['female'] = 0

    # --- MODIFICATION: Create the stacked bar plot ---
    # Plot Male bars first (the bottom layer).
    ax.bar(stacked_data.index, stacked_data['male'], color=gender_colors['male'], label='male', edgecolor='white')
    # Plot female bars on top of the Male bars.
    ax.bar(stacked_data.index, stacked_data['female'], bottom=stacked_data['male'], color=gender_colors['female'], label='female', edgecolor='white')

    ax.set_title(f'Location ID: {loc_id}')
    ax.set_xlabel('Deviations from Mean')
    ax.set_ylabel('Frequency')
    ax.set_xlim(-20.5, 20.5)
    ax.set_xticks(ticks=range(-20, 21, 5))
    ax.grid(axis='y', linestyle='--', alpha=0.7)

# --- Add a single, shared legend for the entire figure ---
handles = [plt.Rectangle((0,0),1,1, color=gender_colors[label]) for label in ['male', 'female']]
labels = ['male', 'female']
fig.legend(handles, labels, title='Gender', loc='center right', bbox_to_anchor=(1.05, 0.5))

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

fig.suptitle('Customer Orders Day Deviation by Gender', fontsize=24, fontweight='bold')
plt.tight_layout(rect=[0, 0.03, 0.95, 0.96])
plt.show()

In [ ]:
loc_id = restaurants_by_4m_coverage[1]
exposure_date = pd.to_datetime(before_after_details_true.loc[loc_id, 'cross_over_date'])
plot_time_series(sales_and_menu_data[loc_id],
                 exposure=exposure_date,
                 freq='D',
                 truncate=False)
plot_time_series((sales_and_menu_data[loc_id]
                  .query('customer_id.isin(@before_after_customers)')
                  ),
                 exposure=exposure_date,
                 freq='D',
                 truncate=False)
plot_time_series_subset(sales_and_menu_data[loc_id],
                 exposure=exposure_date,
                 freq='W',
                 truncate=False,
                 normalize=False)
plot_time_series_subset((sales_and_menu_data[loc_id]
                  .query('customer_id.isin(@before_after_customers)')
                  ),
                 exposure=exposure_date,
                 freq='W',
                 truncate=True,
                 normalize=True)
plt.show()

Aggregate data

In [ ]:
def clip_na(group):
    valid_part = group.dropna(subset='vegan_window_avg')
    first_valid = valid_part.index.min()
    last_valid = valid_part.index.max()
    clipped = group.loc[first_valid:last_valid]
    return clipped

# Aggregate
daily_model_data = (model_data
                    .groupby(['location_id', model_data.index.normalize()], observed=True)
                    .agg({'item_quantity':'sum',
                          'vegan_window_quantity':'sum',
                          'vegetarian_window_quantity':'sum',
                          'meat_window_quantity':'sum',
                          'vegan_window_avg':'mean',
                          'vegetarian_window_avg':'mean',
                          'meat_window_avg':'mean',
                          'vegan_outcome':'sum',
                          'nonvegan_outcome':'sum',
                          'vegetarian_outcome':'sum',
                          'meat_outcome':'sum',})
                    .rename(columns={'item_quantity':'item_quantity_day'})
                    .rename_axis(index=['location_id','created_at'])
                    .reindex(pd.MultiIndex.from_product([location_ids_by_coverage,
                                                         pd.date_range(model_data.index.min().normalize(),
                                                                       model_data.index.max().normalize(),
                                                                       freq='D', tz='UTC')], 
                                                        names=['location_id', 'created_at']))
                    .reset_index()
                    .set_index('created_at')
                    .groupby('location_id', group_keys=False) # Switch argument in future version of Pandas
                    .apply(clip_na)
                    .fillna({'vegan_outcome':0, 'vegetarian_outcome':0, 'nonvegan_outcome':0, 'meat_outcome':0})
                    #.query('0 < item_quantity_day')
                    .assign(
                        vegan_window_avg = lambda df: df.groupby('location_id')['vegan_window_avg'].ffill(),
                        vegetarian_window_avg = lambda df: df.groupby('location_id')['vegetarian_window_avg'].ffill(),
                        meat_window_avg = lambda df: df.groupby('location_id')['meat_window_avg'].ffill(),                        
                        day_of_week_cat = lambda df: df.index.to_series().dt.dayofweek.astype("category"),
                        day_of_week = lambda df: df.index.to_series().dt.dayofweek.astype("category").cat.codes,
                        weekend = lambda df: pd.Series(df.index.dayofweek.isin([5, 6]).astype(int), index=df.index).astype("category"),
                        day_of_month_cat = lambda df: df.index.to_series().dt.day.astype("category"),
                        day_of_month = lambda df: df.index.to_series().dt.day.astype("category").cat.codes,
                        month_cat = lambda df: df.index.to_series().dt.month.astype("category"),
                        month = lambda df: df.index.to_series().dt.month.astype("category").cat.codes,
                        season = lambda df: df.index.month.map(season_from_month).astype("category"),
                        year_cat = lambda df: df.index.to_series().dt.year.astype("category"),
                        year = lambda df: df.index.to_series().dt.year.astype("category").cat.codes,
                        date = lambda df: df.index.to_series().dt.date.astype("category").cat.codes,
                        exposure_VLZX7K2M9QD4T_1 = lambda df: pd.Series(0, index=df.index).mask(
                            (df['location_id'] == 'VLZX7K2M9QD4T') & (pd.to_datetime(before_after_details_true.loc['VLZX7K2M9QD4T', 'cross_over_date']).tz_convert('UTC') <= df.index),
                            1),
                        exposure_SRQS8F7JWA9MZ_1 = lambda df: pd.Series(0, index=df.index).mask(
                            (df['location_id'] == 'SRQS8F7JWA9MZ') & (pd.to_datetime('2020-06-25').tz_localize('UTC') <= df.index),
                            1),
                        exposure_SRQS8F7JWA9MZ_2 = lambda df: pd.Series(0, index=df.index).mask(
                            (df['location_id'] == 'SRQS8F7JWA9MZ') & (pd.to_datetime(before_after_details_true.loc['SRQS8F7JWA9MZ', 'cross_over_date']).tz_convert('UTC') <= df.index),
                            1),
                        exposure_2HRX9P6HKXA8V_1 = lambda df: pd.Series(0, index=df.index).mask(
                            (df['location_id'] == '2HRX9P6HKXA8V') & (pd.to_datetime(before_after_details_true.loc['2HRX9P6HKXA8V', 'cross_over_date']).tz_convert('UTC') <= df.index),
                            1),
                        exposure_JHDN7CF1C03X5_1 = lambda df: pd.Series(0, index=df.index).mask(
                            (df['location_id'] == 'JHDN7CF1C03X5') & (pd.to_datetime(before_after_details_true.loc['JHDN7CF1C03X5', 'cross_over_date']).tz_convert('UTC') <= df.index),
                            1),
                        exposure_JHDN7CF1C03X5_2 = lambda df: pd.Series(0, index=df.index).mask(
                            (df['location_id'] == 'JHDN7CF1C03X5') & (pd.to_datetime('2020-03-12').tz_localize('UTC') <= df.index),
                            1)
                        )
                    )

# Put in location data: locations have data for cuisine, etc.
daily_model_data = pd.merge(daily_model_data, locations, left_on='location_id', right_index=True, how='left')

# Export
daily_model_data.to_parquet("data/5_palate_data_parquet_modeling/all_locations_daily.parquet")

Add in promo and aggregate further (version 1)

In [ ]:
# daily_data_with_promo_list = []
# for loc_id in location_ids:
#     promo_date = before_after_details.loc[loc_id,'cross_over_date'].isocalendar()
#     year = promo_date.year
#     week = promo_date.week
#     day = promo_date.weekday

#     # Treated: after the promo date
#     treated = (weekly_data_with_locations.query(
#         "location_id == @loc_id & ((year == @year & week == @week & day > @day) | "
#         "(year == @year & week > @week) | (year > @year))", engine='python').assign(promo=1))
    
#     # Untreated: before the promo date
#     untreated = (weekly_data_with_locations.query(
#         "location_id == @loc_id & ((year == @year & week == @week & day <= @day) | "
#         "(year == @year & week < @week) | (year < @year))", engine='python').assign(promo=0))
        
#     daily_data_with_promo_list.append(pd.concat([treated,untreated]))

# daily_data_with_promo = pd.concat(daily_data_with_promo_list)

# agg_funcs = {
#     'transactions': 'sum',  # Sum of transactions
#     'all_sales': 'sum',  # Sum of all sales
#     'plant_based_sales': 'sum',  # Sum of plant-based sales
#     'all_items': 'sum',  # Sum of all items
#     'plant_based_items': 'sum',  # Sum of plant-based items
#     'percent_female': 'mean',
#     'percent_items_plant_based': 'mean',  # Average of percent items plant-based
#     'percent_sales_plant_based': 'mean'
# }

# # Group by the required keys and aggregate
# weekly_data_with_promo = daily_data_with_promo.copy().groupby(['location_id', 'year', 'month', 'week', 'cuisine', 'city', 'state',
#        'restaurant_type', 'pos_type', 'zip_code', 'neighborhood_age_5_under',
#        'neighborhood_age_5_9', 'neighborhood_age_10_14',
#        'neighborhood_age_15_19', 'neighborhood_age_20_24',
#        'neighborhood_age_25_29', 'neighborhood_age_30_34',
#        'neighborhood_age_35_39', 'neighborhood_age_40_44',
#        'neighborhood_age_45_49', 'neighborhood_age_50_54',
#        'neighborhood_age_55_59', 'neighborhood_age_60_64',
#        'neighborhood_age_65_69', 'neighborhood_age_70_74',
#        'neighborhood_age_75_79', 'neighborhood_age_80_84',
#        'neighborhood_age_85_89', 'neighborhood_median_hh_income',
#        'neighborhood_race_white', 'neighborhood_race_black',
#        'neighborhood_race_americanIndian_alaskaNative',
#        'neighborhood_race_asian',
#        'neighborhood_race_nativeHawaiian_otherPacificIslander',
#        'neighborhood_race_other', 'batch', 'promo']).agg(agg_funcs).reset_index()

# daily_data_with_promo.to_parquet('data/daily_data.parquet')

Plotting tool

In [ ]:
# Plot function for resampling and visualization
def plot_resampled(data, freq, start_date=None, end_date=None, title="Resampled Predictions"):
    resampled_pred = data.resample(freq)['pred'].mean()
    resampled_actual = data.resample(freq)['vegan_outcome'].mean()

    if start_date and end_date:
        resampled_pred = resampled_pred.loc[start_date:end_date]
        resampled_actual = resampled_actual.loc[start_date:end_date]

    resampled_pred.plot(color='orange', label='Predicted', title=title)
    resampled_actual.plot(color='blue', alpha=0.3, label='Actual', title=title)